# Mercedes F1 Infringement Profile - Abstractive Summarization

This notebook implements abstractive summarization (using DistilBART) for Mercedes F1 infringement documents.

## Objective:
- Process all `no_footer_` files from the preprocessed dataset
- Apply abstractive summarization model (DistilBART - smaller, memory-efficient)
- Maintain temporal analysis (year-by-year summaries)
- Generate abstractive summaries that create new sentences (not just extract)

## Input:
- `pre_proc_op/` folder containing `no_footer_*.txt` files organized by year

## Output:
- Console summaries for each year
- Overall consolidated summary
- Results saved in `distilbart_results/` folder

**Note:** Using DistilBART for better memory efficiency and stability (~260MB vs ~2.3GB for Pegasus).


In [1]:
# Import required libraries
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# DistilBART implementation
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print("Libraries imported successfully")


Libraries imported successfully


In [2]:
# Configuration
processed_base_path = Path("pre_proc_op")
years = ["2020", "2021", "2022", "2023", "2024"]

# Model configuration - Using smaller model to avoid memory issues
# Options:
# - "sshleifer/distilbart-cnn-12-6" (smallest, ~260MB) - RECOMMENDED for stability
# - "facebook/bart-large-cnn" (medium, ~1.6GB)
# - "google/pegasus-xsum" (large, ~2.3GB) - may cause kernel crashes
MODEL_NAME = "sshleifer/distilbart-cnn-12-6"  # Smaller, memory-efficient abstractive summarization model
MAX_LENGTH = 512  # Maximum length for generated summary
MIN_LENGTH = 50   # Minimum length for generated summary
MAX_INPUT_LENGTH = 1024  # Max input tokens (will chunk if needed)

print("Configuration:")
print(f"Processed base path: {processed_base_path}")
print(f"Years to analyze: {years}")
print(f"Model: {MODEL_NAME}")
print(f"Max summary length: {MAX_LENGTH}")
print(f"Min summary length: {MIN_LENGTH}")


Configuration:
Processed base path: pre_proc_op
Years to analyze: ['2020', '2021', '2022', '2023', '2024']
Model: sshleifer/distilbart-cnn-12-6
Max summary length: 512
Min summary length: 50


In [3]:
# Initialize summarizer model
print("Loading summarization model...")
print("This may take a few minutes on first run (downloading model)...")
print(f"Model: {MODEL_NAME} (~260MB - much smaller and more stable)")

try:
    # Use pipeline for simplicity
    summarizer = pipeline(
        "summarization",
        model=MODEL_NAME,
        device=0 if torch.cuda.is_available() else -1,  # Use GPU if available
        framework="pt"
    )
    print(f"Model loaded successfully")
    print(f"Using device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Falling back to CPU...")
    summarizer = pipeline(
        "summarization",
        model=MODEL_NAME,
        device=-1,
        framework="pt"
    )
    print("Model loaded on CPU")


Loading summarization model...
This may take a few minutes on first run (downloading model)...
Model: sshleifer/distilbart-cnn-12-6 (~260MB - much smaller and more stable)


Device set to use cpu


Model loaded successfully
Using device: CPU


In [4]:
# Helper function to chunk long texts
def chunk_text(text, max_chars=4000):
    """
    Split text into chunks that fit within token limits.
    Models typically have max_position_embeddings of 1024, so we chunk conservatively.
    """
    if len(text) <= max_chars:
        return [text]
    
    # Split by sentences (rough approximation)
    sentences = text.split('. ')
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        if len(current_chunk) + len(sentence) + 2 <= max_chars:
            current_chunk += sentence + ". "
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    return chunks

# Helper function to summarize long texts
def summarize_long_text(text, summarizer, max_length=MAX_LENGTH, min_length=MIN_LENGTH):
    """
    Summarize text, handling long inputs by chunking.
    """
    chunks = chunk_text(text)
    
    if len(chunks) == 1:
        # Single chunk - summarize directly
        try:
            result = summarizer(
                text,
                max_length=max_length,
                min_length=min_length,
                do_sample=False
            )
            return result[0]['summary_text']
        except Exception as e:
            print(f"Error in summarization: {e}")
            return ""
    else:
        # Multiple chunks - summarize each and combine
        chunk_summaries = []
        for i, chunk in enumerate(chunks):
            try:
                result = summarizer(
                    chunk,
                    max_length=max_length,
                    min_length=min_length,
                    do_sample=False
                )
                chunk_summaries.append(result[0]['summary_text'])
            except Exception as e:
                print(f"Error summarizing chunk {i+1}/{len(chunks)}: {e}")
                continue
        
        # Combine chunk summaries
        combined = " ".join(chunk_summaries)
        
        # If combined is still too long, summarize again
        if len(combined) > 4000:
            try:
                result = summarizer(
                    combined,
                    max_length=max_length,
                    min_length=min_length,
                    do_sample=False
                )
                return result[0]['summary_text']
            except Exception as e:
                print(f"Error in final summarization: {e}")
                return combined
        
        return combined

print("Helper functions defined")


Helper functions defined


In [5]:
# Process documents year by year
print("MERCEDES F1 INFRINGEMENT PROFILE - ABSTRACTIVE SUMMARIZATION")

yearly_summaries = {}
yearly_stats = {}

for year in years:
    print(f"\nPROCESSING {year} - MERCEDES F1 INFRINGEMENTS")
    
    year_path = processed_base_path / year
    
    if not year_path.exists():
        print(f"Folder {year_path} does not exist")
        continue
    
    # Get all no_footer_ files
    no_footer_files = list(year_path.glob("no_footer_*.txt"))
    
    if not no_footer_files:
        print(f"No no_footer_ files found in {year}")
        continue
    
    print(f"Found {len(no_footer_files)} processed documents in {year}")
    
    # Read and combine all documents for the year
    combined_text = ""
    total_chars = 0
    processed_files = 0
    
    for file_path in no_footer_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read().strip()
                if content:
                    combined_text += content + " "
                    total_chars += len(content)
                    processed_files += 1
        except Exception as e:
            print(f"Error reading {file_path.name}: {e}")
    
    if not combined_text.strip():
        print(f"No valid content found in {year}")
        continue
    
    print(f"Processed {processed_files} files, {total_chars:,} total characters")
    
    # Generate summary using abstractive model
    print(f"\nGenerating abstractive summary for {year}...")
    print("This may take a moment...")
    
    summary = summarize_long_text(
        combined_text.strip(),
        summarizer,
        max_length=MAX_LENGTH,
        min_length=MIN_LENGTH
    )
    
    # Store results
    yearly_summaries[year] = summary
    yearly_stats[year] = {
        'files_processed': processed_files,
        'total_chars': total_chars,
        'summary_length': len(summary)
    }
    
    # Display summary
    print(f"\nABSTRACTIVE SUMMARY FOR {year}:")
    print("-" * 50)
    print(summary)
    print("-" * 50)
    print(f"Summary length: {len(summary):,} characters")
    print(f"Compression ratio: {(len(summary)/total_chars)*100:.1f}%")

print(f"\nProcessed {len(yearly_summaries)} years successfully")


MERCEDES F1 INFRINGEMENT PROFILE - ABSTRACTIVE SUMMARIZATION

PROCESSING 2020 - MERCEDES F1 INFRINGEMENTS
Found 11 processed documents in 2020
Processed 11 files, 10,876 total characters

Generating abstractive summary for 2020...
This may take a moment...

ABSTRACTIVE SUMMARY FOR 2020:
--------------------------------------------------
 The Stewards received a petition from Aston Martin Red Bull Racing (including video footage) for them to review, in accordance with Article 14 of the FIA International Sporting Code, the following decision made by the Stewards at the 2020 Austrian Grand Prix Competition: Document 33 – 2020 Austria Grand Prix . The stewards heard from the driver of Car 44 (Lewis Hamilton) and the team representative and have reviewed video and telemetry evidences . The video footage confirmed that there have been yellow flags and green light panels at the same time and therefore conflicting signals were shown to the driver .  Lewis Hamilton received a team instruction t

Your max_length is set to 512, but your input_length is only 234. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=117)



ABSTRACTIVE SUMMARY FOR 2021:
--------------------------------------------------
 Valtteri Bottas accused of driving unnecessarily slowly in the entry to turn 9 and 10 . Lewis Hamilton disqualified from qualifying after failing to meet the requirements of Art.6.3 of the 2021 FIA Formula 1 Technical Regulations . The Stewards held a hearing on Friday following qualifying with Ron Meadows, the Competitor representative, and Simon Cole, Chief Engineer, Trackside and from the FIA Jo Bauer .  The Stewards decide that car 44 failed the test indicated in TD/011- 19 and is therefore in breach of Art 3.6.3 of the FIA Formula 1 Technical Regulations . The FIA argues that while not regulatory, the TD, like many others, describes the procedure for the test so that competitors may design cars to meet the regulations . However, the Stewards believe that which sections failed is not relevant to the fact that the wing did fail the test .  The driver of car 44, Lewis Hamilton, undid his seat belts on 

Your max_length is set to 512, but your input_length is only 40. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=20)



ABSTRACTIVE SUMMARY FOR 2022:
--------------------------------------------------
 Lewis Hamilton was fined €500 for exceeding the pit lane speed limit by 4.5 km/h . Lewis Hamilton overtook Car 20 when the red light was displayed . Mercedes-AMG Petronas F1 Team fined £1,500 for the breach of Article 37.5 of the FIA Formula One Sporting Regulations .  The Competitor is fined €10,000, which will be suspended for the remainder of the season pending any further violation of the procedure, and the competitors are warned that the passes of the individuals concerned may be revoked in case of systemic violation . The Stewards heard from the driver of Car 44 (Lewis Hamilton) and team representatives, and examined video, radio communication and GPS evidence . Lewis Hamilton was issued a warning for driving unnecessarily slowly during Qualifying .  Stewards heard from the driver of Car 44 (Lewis Hamilton) and team representatives and examined video and radio evidence . Car 55 (Carlos Sainz) was a

Your max_length is set to 512, but your input_length is only 344. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=172)



ABSTRACTIVE SUMMARY FOR 2024:
--------------------------------------------------
 Lewis Hamilton is fined €5,000 for dangerous release of Car 44 . Lewis Hamilton crossed the white line at pit entry when entering the pits . The manoeuvre was not considered dangerous in any way, therefore the least severe penalty available is imposed . The following Power Unit elements have been replaced without approval of the FIA Technical Delegate .  Car 63 (796.5 kg) was below the minimum weight required (798.0 kg) Weight of Car 63 is disqualified from the Race classification . Car 44 (Lewis Hamilton) unnecessarily impeded Car 11 in turn 9 . Mercedes-AMG PETRONAS F1 Team is fined €5,000 .  Lewis Hamilton received a drive through penalty for speeding in the pit lane during a sprint session race . The Mercedes-AMG PETRONAS F1 Team is fined €15,000 for failing to warn their driver of the fact that Car 44 was arriving on a fast lap . Lewis Hamilton was also fined for unnecessarily impeding Car 2 in Turn

In [6]:
# Generate overall consolidated summary
print("\nOVERALL CONSOLIDATED SUMMARY - ALL YEARS")

if yearly_summaries:
    # Combine all yearly summaries
    all_summaries_text = " "
    for year, summary in yearly_summaries.items():
        all_summaries_text += f"{year} Summary: {summary} "
    
    # Generate overall summary
    print("Generating overall consolidated summary...")
    print("This may take a moment...")
    
    overall_summary = summarize_long_text(
        all_summaries_text.strip(),
        summarizer,
        max_length=MAX_LENGTH,
        min_length=MIN_LENGTH
    )
    
    print("\nOVERALL MERCEDES F1 INFRINGEMENT PROFILE (ABSTRACTIVE):")
    print("=" * 50)
    print(overall_summary)
    print("=" * 50)
    
    # Statistics
    total_files = sum(stats['files_processed'] for stats in yearly_stats.values())
    total_chars = sum(stats['total_chars'] for stats in yearly_stats.values())
    
    print(f"\nOVERALL STATISTICS:")
    print(f"Total documents processed: {total_files}")
    print(f"Total characters analyzed: {total_chars:,}")
    print(f"Overall summary length: {len(overall_summary):,} characters")
    print(f"Overall compression ratio: {(len(overall_summary)/total_chars)*100:.1f}%")
    
    print(f"\nYEARLY BREAKDOWN:")
    for year, stats in yearly_stats.items():
        compression = (stats['summary_length']/stats['total_chars'])*100 if stats['total_chars'] > 0 else 0
        print(f"{year}: {stats['files_processed']} docs, {stats['total_chars']:,} chars -> {stats['summary_length']:,} chars ({compression:.1f}%)")
    
else:
    print("No summaries generated. Please check the input files.")



OVERALL CONSOLIDATED SUMMARY - ALL YEARS
Generating overall consolidated summary...
This may take a moment...



OVERALL MERCEDES F1 INFRINGEMENT PROFILE (ABSTRACTIVE):
 Mercedes-AMG Petronas F1 Team fined €25,000 for practice start violation . Lewis Hamilton received a team instruction to perform the practice start in the incorrect place . Valtteri Bottas collided with the rear of car 4 in the braking zone to turn 1 . Hamilton was reprimanded for failing to follow the Race Director’s instruction . Hamilton admitted wearing an item of jewellery during the session .  Mercedes-AMG Petronas F1 Team's Lewis Hamilton fined €100 for wearing of jewellery . Lewis Hamilton received a warning from the FIA for leaving the track without a justifiable reason multiple times in practice . The British driver was fined £100 for exceeding the pit lane speed limit by 0.2 km/h . The Stewards reviewed the video evidence and determined that the driver of car 63 braked late into turn 1 and collided with car 55, and was therefore wholly to blame for the collision . Mercedes team doctor requested exemption from complian

In [7]:
# Save results to files
print("\nSAVING RESULTS")

# Create output directory
output_dir = Path("distilbart_results")
output_dir.mkdir(exist_ok=True)

# Save yearly summaries
for year, summary in yearly_summaries.items():
    output_file = output_dir / f"distilbart_summary_{year}.txt"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f"Mercedes F1 Infringement Summary - {year}\n")
        f.write("="*50 + "\n\n")
        f.write(summary)
    print(f"Saved {year} summary to {output_file}")

# Save overall summary
if yearly_summaries:
    overall_file = output_dir / "distilbart_overall_summary.txt"
    with open(overall_file, 'w', encoding='utf-8') as f:
        f.write("Mercedes F1 Infringement Profile - Overall Summary (Abstractive)\n")
        f.write("="*50 + "\n\n")
        f.write(overall_summary)
        f.write("\n\n" + "="*50 + "\n")
        f.write("STATISTICS\n")
        f.write("="*50 + "\n")
        total_files = sum(stats['files_processed'] for stats in yearly_stats.values())
        total_chars = sum(stats['total_chars'] for stats in yearly_stats.values())
        f.write(f"Total documents processed: {total_files}\n")
        f.write(f"Total characters analyzed: {total_chars:,}\n")
        f.write(f"Overall summary length: {len(overall_summary):,} characters\n")
        f.write(f"Overall compression ratio: {(len(overall_summary)/total_chars)*100:.1f}%\n")
    print(f"Saved overall summary to {overall_file}")

print("\nAll results saved successfully!")



SAVING RESULTS
Saved 2020 summary to distilbart_results\distilbart_summary_2020.txt
Saved 2021 summary to distilbart_results\distilbart_summary_2021.txt
Saved 2022 summary to distilbart_results\distilbart_summary_2022.txt
Saved 2023 summary to distilbart_results\distilbart_summary_2023.txt
Saved 2024 summary to distilbart_results\distilbart_summary_2024.txt
Saved overall summary to distilbart_results\distilbart_overall_summary.txt

All results saved successfully!
